In [37]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cna, glob, os

In [67]:
def test_clusters(df, cols, pheno, donor, Nnull=1000):
    X = df[cols].values.copy()
    pheno = df[pheno].copy()
    donorids = df[donor].copy()
    
    X = (X - X.mean(axis=0))/X.std(axis=0)
    y_ = cna.tl._stats.grouplevel_permutation(donorids, pheno, Nnull).astype('float')
    pheno = (pheno - pheno.mean())/pheno.std()
    y_ -= y_.mean(axis=0)
    y_ /= y_.std(axis=0)
    
    ncorrs = X.T.dot(pheno) / len(X)
    nullncorrs = X.T.dot(y_) / len(X)
    pvals = ((np.abs(nullncorrs) >= np.abs(ncorrs)[:,None]).sum(axis=1) + 1)/(Nnull + 1)
    
    maxcorr = max(np.abs(ncorrs).max(), 0.001)
    fdr_thresholds = np.arange(maxcorr/4, maxcorr, maxcorr/400)
    fdr_vals = cna.tl._stats.empirical_fdrs(ncorrs, nullncorrs, fdr_thresholds)

    fdrs = pd.DataFrame({
        'threshold':fdr_thresholds,
        'fdr':fdr_vals,
        'num_detected': [(np.abs(ncorrs)>t).sum() for t in fdr_thresholds]})
    return fdrs, ncorrs, pvals

In [68]:
from scipy.stats import entropy
def assess_integration(d):
    NAM, _ = cna.tl.nam(d, 'sid')
    NAM /= NAM.sum(axis=0)
    d.obs['isi'] = np.power(2, entropy(NAM, axis=0) / np.log(2))
    # plt.figure(figsize=(3,1.5))
    # plt.hist(d.obs.isi, bins=50)
    # plt.gca().set_yscale('log')
    # plt.xlim(0, len(d.obs.sid.unique()))
    # plt.show()

def test_cluster_cc(d, samplemeta):
    if 'cluster_method' in d.obs.columns:
        cluster_key = 'cluster_method'
    elif 'leiden_1' in d.obs.columns:
        cluster_key = 'leiden_1'
    else:
        cluster_key = [c for c in d.obs.columns if c.startswith('leiden')][-1] #TODO make it choose highest res
    
    print(f'Using {cluster_key} for clustering')
    ct = pd.crosstab(
        d.obs['sid'], d.obs[cluster_key], 
    )
    ct.index.name = 'sid'
    ct = ct.div(ct.sum(axis=1), axis=0)
    clusts = ct.columns.values
    ct['donor'] = samplemeta['donor']
    ct['case'] = samplemeta.case

    fdrs, stats, ps = test_clusters(ct, clusts, 'case', 'donor', Nnull=10000)
    d.uns['clustercc'] = pd.DataFrame(
        {'corr':stats, 'p':ps},
        index=pd.Series(clusts, name='cluster'),
    )
    d.uns['clustercc_key'] = cluster_key
    d.uns['clustercc_minp'] = np.min(ps*len(ps))

    # plt.figure(figsize=(3,1.5))
    # plt.scatter(stats, -np.log10(ps))
    # plt.axhline(-np.log10(0.05/len(ps)), ls='--')
    # plt.axvline(0, color='k')
    # plt.xlabel('Corr. to case status', fontsize=10)
    # plt.ylabel('$-\\log_{10}(P)$', fontsize=10)
    # plt.gca().spines[['right', 'top']].set_visible(False)
    # plt.show()

def test_mn_cc(d, samplemeta):
    d.uns['mncc_p'] = cna.tl.association(d, samplemeta.case, 'sid', donorids=samplemeta.donor, key_added='mncoef')
    d.uns['mncc_npos'] = ((d.obs.mncoef_fdr <= 0.1) & (d.obs.mncoef > 0)).sum()
    d.uns['mncc_nneg'] = ((d.obs.mncoef_fdr <= 0.1) & (d.obs.mncoef < 0)).sum()

In [69]:
def assess(dsetname, samplemeta):
    embeddings = glob.glob(f'_embeddings/{dsetname}*.h5ad')
    for embedding in embeddings:
        fname = os.path.basename(embedding)
        method = fname.split('_')[1]
        harm = fname.split('_')[2]
        print(method, harm)
        d = sc.read_h5ad(embedding)
        assess_integration(d)
        test_cluster_cc(d, samplemeta)
        test_mn_cc(d, samplemeta)
        print(method, harm, d.obs.isi.median(), d.uns['clustercc_minp'],
              d.uns['mncc_p'], d.uns['mncc_npos'], d.uns['mncc_nneg'])
        print('======')
        d.write(f'_results/{fname}')

# ALZ

In [31]:
# generate samplemeta with one row per sample (rather than per donor)
cells = pd.read_csv('../../ALZ/alz-data/SEAAD_MTG_MERFISH_metadata.2024-05-03.noblanks.harmonized.txt',
                         sep='\t')
cells['donor'] = cells.index.str.split('_').str[0]
cells['sid'] = cells.index.str.split('_').str[1]
sid_to_donor = cells[['sid', 'donor']].drop_duplicates()

samplemeta = pd.read_csv('../../ALZ/alz-data/sea-ad_cohort_donor_metadata_encoded_20240924.tsv',
                         sep='\t').drop(columns=['Donor ID']).set_index('donor', drop=True)
samplemeta = pd.merge(sid_to_donor, samplemeta, left_on='donor', right_index=True, how='left').set_index('sid', drop=True)
samplemeta['case'] = samplemeta['Consensus Clinical Dx (choice=Control)'] != 'Checked'

In [70]:
assess('ALZ', samplemeta)

patchavgmm noharm.h5ad
Using leiden_1 for clustering


/Users/yakir/Dropbox/py/cna/src/cna/tools/_stats.py:26: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Yg = np.array([Y[G==g][0] for g in Gu])


patchavgmm noharm.h5ad 9.712073278895147 0.1663833616638336 0.7442557442557443 0 0
utag harm.h5ad
Using leiden_0.1 for clustering


/Users/yakir/Dropbox/py/cna/src/cna/tools/_stats.py:26: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Yg = np.array([Y[G==g][0] for g in Gu])


utag harm.h5ad 34.952991075154415 0.43515648435156484 0.4805194805194805 0 0
stagate harm.h5ad
Using leiden_1 for clustering


...

canvas noharm.h5ad 18.90088520606384 1.0617938206179383 0.2217782217782218 0 0
patchcelltypeabundance harm.h5ad
Using leiden_1 for clustering


/Users/yakir/Dropbox/py/cna/src/cna/tools/_stats.py:26: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Yg = np.array([Y[G==g][0] for g in Gu])


patchcelltypeabundance harm.h5ad 30.559845258679555 0.06719328067193281 0.029970029970029972 11 2879
patchavgmm harm.h5ad
Using leiden_1 for clustering


/Users/yakir/Dropbox/py/cna/src/cna/tools/_stats.py:26: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Yg = np.array([Y[G==g][0] for g in Gu])


patchavgmm harm.h5ad 26.115122926946725 0.03999600039996 0.08091908091908091 0 0
